# Batch Full-Text Extraction Evaluation

Evaluate full-text extraction using GROBID-parsed PDFs across all Fuster validation samples.

**Approach:**
- Parse PDFs with GROBID to extract all sections
- Build full-text prompts using all sections
- Run GPT-5-mini extraction with low reasoning
- Compare against manually annotated ground truth
- Use `DatasetFeaturesNormalized` for ground truth validation

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Project modules
from llm_metadata.fulltext_pipeline import (
    FulltextPipelineConfig,
    SectionSelectionConfig,
    GPTClassifyConfig,
    FulltextInputRecord,
    FulltextOutputRecord,
    ParsedDocumentRecord,
    PromptRecord,
    # Combined flow
    fulltext_extraction_pipeline,
    # Staged flows (separate parallelization)
    staged_fulltext_pipeline,
    grobid_parsing_flow,
    prompt_building_flow,
    gpt_classification_flow,
    # Single PDF
    process_single_pdf,
    # Utilities
    load_input_manifest,
    save_output_manifest,
    build_fulltext_prompt,
    collect_relevant_sections,
)
from llm_metadata.pdf_parsing import process_pdf, ParsedDocument, Section
from llm_metadata.gpt_classify import classify_abstract
from llm_metadata.chunking import count_tokens
from llm_metadata.section_normalize import classify_section
from llm_metadata.schemas.chunk_metadata import SectionType
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.schemas.evaluation import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# Paths
PDF_DIR = Path("data/pdfs/fuster")
MANIFEST_PATH = PDF_DIR / "manifest.csv"
GROUND_TRUTH_PATH = Path("data/dataset_092624.xlsx")
OUTPUT_DIR = Path("artifacts/fulltext_results")

print(f"PDF directory: {PDF_DIR.absolute()}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")
print(f"Ground truth exists: {GROUND_TRUTH_PATH.exists()}")

## Step 1: Load Manifest and Ground Truth

Map dataset DOIs to article DOIs and load ground truth annotations.

In [ ]:
# Load manifest with DOI mapping
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest_df)} total rows")

# Filter to downloaded PDFs only
downloaded_df = manifest_df[manifest_df['status'] == 'downloaded'].copy()
print(f"Downloaded PDFs: {len(downloaded_df)}")

# Create DOI mapping (dataset_doi -> article_doi)
doi_mapping = dict(zip(downloaded_df['dataset_doi'], downloaded_df['article_doi']))
print(f"DOI mappings: {len(doi_mapping)}")

# Display sample
downloaded_df[['article_doi', 'dataset_doi', 'downloaded_pdf_path']].head(5)

In [ ]:
# Load ground truth
gt_df = pd.read_excel(GROUND_TRUTH_PATH)
print(f"Ground truth: {len(gt_df)} total rows")

# Extract dataset DOI from URL column
gt_df['dataset_doi'] = gt_df['url'].str.replace('https://doi.org/', '', regex=False)

# Filter to DOIs that have downloaded PDFs
gt_matched = gt_df[gt_df['dataset_doi'].isin(doi_mapping.keys())].copy()
print(f"Ground truth with matched PDFs: {len(gt_matched)}")

# Add article DOI mapping
gt_matched['article_doi'] = gt_matched['dataset_doi'].map(doi_mapping)

gt_matched[['dataset_doi', 'article_doi', 'data_type', 'species']].head(5)

In [ ]:
# Validate ground truth with DatasetFeaturesNormalized
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset'
]

ground_truth_by_dataset = {}
validation_errors = []

for _, row in gt_matched.iterrows():
    dataset_doi = row['dataset_doi']
    try:
        # Extract relevant columns and convert to dict
        row_dict = {col: row[col] for col in RELEVANT_COLS if col in row.index}
        
        # Validate using DatasetFeaturesNormalized
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_dataset[dataset_doi] = validated
    except Exception as e:
        validation_errors.append({'doi': dataset_doi, 'error': str(e)})

print(f"Successfully validated: {len(ground_truth_by_dataset)} records")
print(f"Validation errors: {len(validation_errors)} records")

if validation_errors:
    print("\nFirst 3 errors:")
    for err in validation_errors[:3]:
        print(f"  {err['doi']}: {err['error'][:100]}...")

## Step 2: Configure Pipeline

Set up full-text extraction with all sections, gpt-5-mini, and low reasoning.

In [ ]:
# Pipeline configuration
config = FulltextPipelineConfig(
    section_config=SectionSelectionConfig(
        include_all=True,  # Use ALL sections from PDFs
        include_abstract=True,
    ),
    gpt_config=GPTClassifyConfig(
        model="gpt-5-mini",
        reasoning={"effort": "low"},
        max_output_tokens=4096,
    ),
    grobid_url="http://localhost:8070",
    pdf_dir=PDF_DIR,
    text_format=DatasetFeatures,
)

print("Pipeline Configuration:")
print(f"  Model: {config.gpt_config.model}")
print(f"  Reasoning: {config.gpt_config.reasoning}")
print(f"  Section selection: include_all={config.section_config.include_all}")
print(f"  GROBID URL: {config.grobid_url}")

## Step 3: Run Full-Text Extraction

Process all PDFs through GROBID parsing and GPT classification.

In [ ]:
# Build input records for PDFs with ground truth
input_records = []
for _, row in gt_matched.iterrows():
    dataset_doi = row['dataset_doi']
    article_doi = row['article_doi']
    
    # Find PDF path from manifest
    manifest_row = downloaded_df[downloaded_df['dataset_doi'] == dataset_doi]
    if manifest_row.empty:
        continue
    
    pdf_path = PDF_DIR / manifest_row.iloc[0]['downloaded_pdf_path']
    if not pdf_path.exists():
        # Try alternate path construction
        pdf_path = PDF_DIR.parent / manifest_row.iloc[0]['downloaded_pdf_path']
    
    if pdf_path.exists():
        input_records.append(FulltextInputRecord(
            article_doi=article_doi,
            dataset_doi=dataset_doi,
            pdf_path=str(pdf_path),
        ))

print(f"Input records to process: {len(input_records)}")

In [ ]:
# Process PDFs using STAGED PIPELINE with separate parallelization
# Stage 1: GROBID parsing (max_workers=10) - GROBID handles concurrency well
# Stage 2: Prompt building (max_workers=5) - CPU-bound
# Stage 3: GPT classification (max_workers=5) - API rate limits

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_manifest_path = OUTPUT_DIR / f"fulltext_results_{timestamp}.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running STAGED pipeline on {len(input_records)} PDFs...")
print(f"Using {config.gpt_config.model} with {config.gpt_config.reasoning}")
print()

# Run the staged Prefect flow with separate parallelization per stage
results = staged_fulltext_pipeline(
    input_records=input_records,
    config=config,
    output_manifest=output_manifest_path,
    grobid_workers=10,  # Higher parallelization for GROBID
    gpt_workers=5,       # Controlled parallelization for GPT API
)

# Summary
success_count = sum(1 for r in results if r.status == "success")
error_count = sum(1 for r in results if r.status == "error")
skipped_count = sum(1 for r in results if r.status == "skipped")
print(f"\nProcessing complete: {success_count} success, {error_count} errors, {skipped_count} skipped")
print(f"Results saved to: {output_manifest_path}")

## Step 4: Prepare Extractions for Evaluation

Convert extraction results to Pydantic models for evaluation.

In [ ]:
# Build predictions by dataset DOI
predictions_by_dataset = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    
    dataset_doi = result.dataset_doi
    if not dataset_doi:
        continue
    
    try:
        # Create DatasetFeatures from extraction dict
        pred = DatasetFeatures.model_validate(result.extraction)
        predictions_by_dataset[dataset_doi] = pred
    except Exception as e:
        print(f"Error validating prediction for {dataset_doi}: {e}")

print(f"Valid predictions: {len(predictions_by_dataset)}")

# Find common DOIs between predictions and ground truth
common_dois = set(predictions_by_dataset.keys()) & set(ground_truth_by_dataset.keys())
print(f"Common DOIs for evaluation: {len(common_dois)}")

## Step 5: Evaluate Full-Text Extraction

Compare full-text extractions against ground truth using evaluation framework.

In [ ]:
# Evaluation fields
EVAL_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f', 'species'
]

# Evaluation configuration with fuzzy matching for species
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    fuzzy_match_fields={
        'species': FuzzyMatchConfig(threshold=80),
    }
)

# Filter to common DOIs
true_by_id = {doi: ground_truth_by_dataset[doi] for doi in common_dois}
pred_by_id = {doi: predictions_by_dataset[doi] for doi in common_dois}

# Run evaluation
fulltext_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("FULL-TEXT Extraction Metrics:")
print("=" * 70)
display(fulltext_report.metrics_to_pandas())

In [ ]:
# Aggregate metrics
ft_micro = micro_average(fulltext_report.field_metrics.values())
ft_macro = macro_f1(fulltext_report.field_metrics.values())

print("\nAggregate Metrics:")
print("=" * 50)
print(f"{'Metric':<25} {'Value':>15}")
print("-" * 50)
print(f"{'Micro Precision':<25} {ft_micro.precision or 0:>15.3f}")
print(f"{'Micro Recall':<25} {ft_micro.recall or 0:>15.3f}")
print(f"{'Micro F1':<25} {ft_micro.f1 or 0:>15.3f}")
print(f"{'Macro F1':<25} {ft_macro or 0:>15.3f}")
print(f"{'Records Evaluated':<25} {len(common_dois):>15}")
print("=" * 50)

## Step 6: Run Abstract-Only Baseline (Optional)

Compare full-text extraction against abstract-only baseline.

In [ ]:
# Run abstract-only extraction for comparison
RUN_BASELINE = True  # Set to False to skip baseline comparison

if RUN_BASELINE:
    abstract_predictions = {}
    
    for record in tqdm(input_records, desc="Abstract-only extraction"):
        if record.dataset_doi not in common_dois:
            continue
        
        try:
            # Parse PDF to get abstract
            work_id = record.article_doi.replace("/", "_")
            _, doc = process_pdf(
                pdf_path=Path(record.pdf_path),
                work_id=work_id,
                grobid_url=config.grobid_url,
            )
            
            if not doc.abstract:
                continue
            
            # Run classification with abstract only
            result = classify_abstract(
                abstract=doc.abstract,
                text_format=DatasetFeatures,
                model=config.gpt_config.model,
                reasoning=config.gpt_config.reasoning,
                max_output_tokens=config.gpt_config.max_output_tokens,
            )
            
            if result.get("output"):
                abstract_predictions[record.dataset_doi] = result["output"]
                
        except Exception as e:
            print(f"Error: {record.dataset_doi}: {e}")
    
    print(f"\nAbstract-only predictions: {len(abstract_predictions)}")

In [ ]:
if RUN_BASELINE and abstract_predictions:
    # Evaluate abstract-only
    abstract_common = set(abstract_predictions.keys()) & set(ground_truth_by_dataset.keys())
    
    abstract_true = {doi: ground_truth_by_dataset[doi] for doi in abstract_common}
    abstract_pred = {doi: abstract_predictions[doi] for doi in abstract_common}
    
    abstract_report = evaluate_indexed(
        true_by_id=abstract_true,
        pred_by_id=abstract_pred,
        fields=EVAL_FIELDS,
        config=eval_config,
    )
    
    ab_micro = micro_average(abstract_report.field_metrics.values())
    ab_macro = macro_f1(abstract_report.field_metrics.values())
    
    # Comparison table
    print("\nCOMPARISON: Full-Text vs Abstract-Only")
    print("=" * 70)
    print(f"{'Metric':<25} {'Full-text':>15} {'Abstract':>15} {'Delta':>12}")
    print("-" * 70)
    print(f"{'Micro Precision':<25} {ft_micro.precision or 0:>15.3f} {ab_micro.precision or 0:>15.3f} {(ft_micro.precision or 0) - (ab_micro.precision or 0):>+12.3f}")
    print(f"{'Micro Recall':<25} {ft_micro.recall or 0:>15.3f} {ab_micro.recall or 0:>15.3f} {(ft_micro.recall or 0) - (ab_micro.recall or 0):>+12.3f}")
    print(f"{'Micro F1':<25} {ft_micro.f1 or 0:>15.3f} {ab_micro.f1 or 0:>15.3f} {(ft_micro.f1 or 0) - (ab_micro.f1 or 0):>+12.3f}")
    print(f"{'Macro F1':<25} {ft_macro or 0:>15.3f} {ab_macro or 0:>15.3f} {(ft_macro or 0) - (ab_macro or 0):>+12.3f}")
    print("=" * 70)

## Step 7: Per-Field Analysis

In [ ]:
# Per-field comparison
if RUN_BASELINE and abstract_predictions:
    comparison_data = []
    
    for field in EVAL_FIELDS:
        ft_m = fulltext_report.field_metrics.get(field)
        ab_m = abstract_report.field_metrics.get(field)
        
        ft_f1 = ft_m.f1 if ft_m and ft_m.f1 else 0
        ab_f1 = ab_m.f1 if ab_m and ab_m.f1 else 0
        
        comparison_data.append({
            'field': field,
            'fulltext_precision': round(ft_m.precision or 0, 3) if ft_m else 0,
            'fulltext_recall': round(ft_m.recall or 0, 3) if ft_m else 0,
            'fulltext_f1': round(ft_f1, 3),
            'abstract_precision': round(ab_m.precision or 0, 3) if ab_m else 0,
            'abstract_recall': round(ab_m.recall or 0, 3) if ab_m else 0,
            'abstract_f1': round(ab_f1, 3),
            'delta_f1': round(ft_f1 - ab_f1, 3),
            'winner': 'FULL-TEXT' if ft_f1 > ab_f1 else ('ABSTRACT' if ab_f1 > ft_f1 else 'TIE')
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print("\nPer-Field Comparison:")
    display(comparison_df)
    
    # Summary
    ft_wins = sum(1 for r in comparison_data if r['winner'] == 'FULL-TEXT')
    ab_wins = sum(1 for r in comparison_data if r['winner'] == 'ABSTRACT')
    ties = sum(1 for r in comparison_data if r['winner'] == 'TIE')
    print(f"\nFields won: FULL-TEXT={ft_wins}, ABSTRACT={ab_wins}, TIE={ties}")

## Step 8: Cost Analysis

In [ ]:
# Cost analysis from results
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens = sum(r.output_tokens or 0 for r in successful_results)
    total_fulltext_tokens = sum(r.fulltext_tokens or 0 for r in successful_results)
    total_abstract_tokens = sum(r.abstract_tokens or 0 for r in successful_results)
    
    print("\nCOST ANALYSIS")
    print("=" * 50)
    print(f"{'Metric':<30} {'Value':>18}")
    print("-" * 50)
    print(f"{'Total PDFs Processed':<30} {len(successful_results):>18}")
    print(f"{'Avg Abstract Tokens':<30} {total_abstract_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Full-text Tokens':<30} {total_fulltext_tokens / len(successful_results):>18,.0f}")
    print(f"{'Token Increase (avg)':<30} {(total_fulltext_tokens - total_abstract_tokens) / len(successful_results):>18,.0f}")
    print("-" * 50)
    print(f"{'Total Input Tokens':<30} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<30} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<30} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per PDF (USD)':<30} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 50)

## Step 9: Error Analysis

In [ ]:
# Show sample of mismatches for each field
print("SAMPLE ERROR ANALYSIS")
print("=" * 80)

for field in EVAL_FIELDS[:3]:  # First 3 fields
    errors = fulltext_report.errors_for_field(field)
    if errors:
        print(f"\n{field} ({len(errors)} errors):")
        print("-" * 60)
        for err in errors[:2]:  # First 2 errors per field
            print(f"  DOI: {err.record_id}")
            print(f"    Ground Truth: {err.true_value}")
            print(f"    Prediction:   {err.pred_value}")
            print()

## Conclusions

### Results Summary

| Metric | Full-text | Abstract-only | Delta |
|--------|-----------|---------------|-------|
| Micro F1 | TBD | TBD | TBD |
| Macro F1 | TBD | TBD | TBD |
| Total Cost | TBD | TBD | TBD |

### Key Findings

1. **TBD** - Does full-text improve overall extraction quality?
2. **TBD** - Which fields benefit most from full-text?
3. **TBD** - Is the additional cost justified by performance gains?

### Recommendations

TBD after running notebook.